#PARTE 2: Impacto Financiero y JOINS

11. El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT 
    count(*) AS Total_reservas_canceladas_TATO,
    total_reservas_canceladas_tato*1.5 AS TotalDinero_TATO
FROM cursodataacademy.project_hotel.tb_reservas
WHERE distribution_channel LIKE 'TA/TO' AND  is_canceled LIKE '1';

`Conclusion 11:` El hotel pagó USD 60,228 a las agencias TA/TO por 40,152 reservas que terminaron canceladas, es decir, dinero sin ningún ingreso a cambio.
Esas cancelaciones equivalen a cerca del 41% de las 97,870 reservas del canal, lo que muestra que TA/TO es el canal más grande y también uno de los más riesgosos.
Aunque la comisión por reserva es pequeña, el volumen convierte esta pérdida en un costo recurrente que se puede reducir.

12. El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
SELECT 
    count(*) AS Cant_Cancel,
    cant_cancel*120 AS Perd_Total
FROM cursodataacademy.project_hotel.tb_reservas
WHERE date_diff(day, to_date(reservation_status_date, 'd/M/yy'), arrival_date) < 3
        AND is_canceled LIKE '1';

`Conclusion 12:` Un total de 3204 reservas se cancelaron con menos de 3 días de anticipación, lo que equivale a una pérdida de S/ 384480.
Son cancelaciones de último momento, cuando el hotel casi no tiene tiempo de revender la habitación, y por eso son las más costosas.
Esta pérdida es unas 6 veces mayor que los S/60228 pagados en comisiones a TA/TO por reservas canceladas

13. (JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql 
SELECT
    r.is_canceled,
    p.nombre_pais
FROM cursodataacademy.project_hotel.tb_reservas AS r
LEFT JOIN cursodataacademy.project_hotel.tb_paises AS p
ON r.country = p.cod_pais;


`Conclusion 13:` El LEFT JOIN une cada reserva con el nombre de su país, de modo que el reporte muestra “Portugal” o “Reino Unido” en lugar de códigos como PRT o GBR.
Al conservar todas las reservas, el equipo comercial puede analizar el origen de los clientes sin perder registros que no tengan país asociado.

14. (JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?


In [0]:
%sql           
SELECT 
    p.nombre_pais AS Paises,
    count(1) AS Paises_Reservas_Canceladas
FROM cursodataacademy.project_hotel.tb_reservas AS r
INNER JOIN cursodataacademy.project_hotel.tb_paises AS p
ON r.country = p.cod_pais
WHERE r.is_canceled LIKE '1'
GROUP BY p.nombre_pais
ORDER BY paises_reservas_canceladas DESC;

`Conclusion 14:` Portugal concentra 27519 cancelaciones, es decir, cerca del 62% de las 44224 del hotel, muy por encima del resto.
Le siguen Reino Unido, España, Francia e Italia.
Portugal no solo cancela más en cifras absolutas, según la base de datos, más de la mitad de sus reservas se cancelan, seguido de Reino Unido España y Francia.
Esto indica que el problema de cancelaciones se concentra en el mercado europeo.

15. (JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.

In [0]:
%sql   
SELECT 
    r.distribution_channel,
    v.nombre_vendedor
FROM cursodataacademy.project_hotel.tb_reservas AS r
FULL OUTER JOIN cursodataacademy.project_hotel.tb_vendedores AS v
ON r.agent = v.id_vendedor;


`Conclusion 15:`El FULL OUTER JOIN cruza cada reserva con su vendedor, y en las filas visibles todasla gran mayoria son del canal TA/TO, atendidas por los agentes 006 y 196.
En cambio, 3,167 reservas de TA/TO tampoco tienen agente, lo que sí es un vacío de registro y conviene revisar con el área comercial.
Los 333 vendedores de la tabla tienen reservas asignadas, así que no habría vendedores sin reservas .

16. (JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
%sql      /*                                                           
SELECT 
    f.monto_facturado,
    r.hotel
FROM cursodataacademy.project_hotel.tb_reservas AS r
RIGHT JOIN cursodataacademy.project_hotel.tb_facturas AS f
ON r.id_reserva = f.id_reserva;*/